# Early detection at scale (GPU)

Bi-GCN trained on full cascades (**bert** features), evaluated on test cascades truncated to their first 20--100% of nodes (BFS order from the news root, same logic as `src/factnet/graph/early_detection.py`), on both UPFD datasets.

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Run all` (~10 min). The last cell saves `early_detection_metrics.json`.

In [ ]:
%pip install -q torch-geometric

import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Bi-GCN, identical to src/factnet/graph/models.py
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GCNConv, global_mean_pool


class BiGCN(torch.nn.Module):
    """Bidirectional GCN: top-down on the propagation edges, bottom-up on the
    reversed edges, pooled and concatenated (Bian et al., 2020)."""

    def __init__(self, in_dim, hidden, num_classes):
        super().__init__()
        self.td1 = GCNConv(in_dim, hidden)
        self.td2 = GCNConv(hidden, hidden)
        self.bu1 = GCNConv(in_dim, hidden)
        self.bu2 = GCNConv(hidden, hidden)
        self.lin = Linear(hidden * 2, num_classes)

    def forward(self, x, edge_index, batch):
        rev = edge_index.flip(0)
        td = F.relu(self.td1(x, edge_index))
        td = F.relu(self.td2(td, edge_index))
        bu = F.relu(self.bu1(x, rev))
        bu = F.relu(self.bu2(bu, rev))
        td = global_mean_pool(td, batch)
        bu = global_mean_pool(bu, batch)
        return self.lin(torch.cat([td, bu], dim=1))

In [ ]:
# BFS truncation + train/eval, same protocol as the local CPU study
import json
import math
from collections import deque

from sklearn.metrics import f1_score
from torch_geometric.data import Data
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.utils import subgraph

FRACTIONS = (0.2, 0.4, 0.6, 0.8, 1.0)


def bfs_order(data):
    adj = [[] for _ in range(data.num_nodes)]
    for s, t in data.edge_index.t().tolist():
        adj[s].append(t)
        adj[t].append(s)
    seen, order, queue = {0}, [], deque([0])
    while queue:
        node = queue.popleft()
        order.append(node)
        for nxt in sorted(adj[node]):
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    order += sorted(set(range(data.num_nodes)) - set(order))
    return order


def truncate(data, fraction):
    if fraction >= 1.0:
        return data
    order = bfs_order(data)
    keep = torch.tensor(sorted(order[: max(1, math.ceil(fraction * len(order)))]))
    edge_index, _ = subgraph(keep, data.edge_index, relabel_nodes=True,
                             num_nodes=data.num_nodes)
    return Data(x=data.x[keep], edge_index=edge_index, y=data.y)


def train_bigcn(train_ds, epochs=60, hidden=64, lr=0.01, seed=0):
    torch.manual_seed(seed)
    model = BiGCN(train_ds.num_features, hidden, train_ds.num_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    for _ in range(epochs):
        model.train()
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            F.cross_entropy(out, batch.y).backward()
            opt.step()
    return model


def evaluate(model, dataset, fraction):
    loader = DataLoader([truncate(d, fraction) for d in dataset], batch_size=128)
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch.x, batch.edge_index, batch.batch).argmax(dim=1)
            ps += pred.cpu().tolist()
            ys += batch.y.cpu().tolist()
    return f1_score(ys, ps, average="macro")


results = []
for name in ("politifact", "gossipcop"):
    train_ds = UPFD("/tmp/upfd", name, "bert", split="train")
    test_ds = UPFD("/tmp/upfd", name, "bert", split="test")
    model = train_bigcn(train_ds)
    print(f"\n=== {name} / bert  (test {len(test_ds)}) ===")
    for fraction in FRACTIONS:
        f1 = evaluate(model, test_ds, fraction)
        print(f"  {fraction:>5.0%}  macro-F1 {f1:.3f}")
        results.append({"dataset": name, "fraction": fraction,
                        "macro_f1": round(f1, 3)})

with open("early_detection_metrics.json", "w") as f:
    json.dump({"runs": results}, f, indent=2)

try:
    from google.colab import files
    files.download("early_detection_metrics.json")
except ImportError:
    print("saved early_detection_metrics.json")